# Place Matching Pipeline

Matches place names produced by knowledge-extraction processes to geocoded entries in one or more place databases (gazetteers). Input sources and gazetteers are interchangeable components. Routing rules select which gazetteers apply to which books, place kinds and years. A separate enrichment step links matched points to historical census subdivisions.

Status: first version. Matching rules, weights and thresholds are defined in the configuration cell and are expected to be tuned against the review output.

## 0. Requirements, inputs and outputs

### Environment
- Python 3.10 or later, run locally in Jupyter.
- Packages: `pandas`, `numpy`, `geopandas` (1.0 or later), `shapely` (2.0 or later), `pyogrio`. `tqdm` is optional (progress bars).
- `pyogrio` reads the CHGIS file geodatabase through GDAL's OpenFileGDB driver. No ESRI software is required.

### Network access
None. All gazetteers are read from local files.

### Disk access
Reads the input and gazetteer paths defined in the paths cell. Writes to `OUTPUT_DIR` only.

### Data the notebook reads

| Item | Path variable | Format | Required |
|---|---|---|---|
| KG location nodes | `KG_FILE_PATTERNS["locations"]` | `<book>_nodes_locations.json`: JSON array of node records | If `kg_v3` is in `ACTIVE_INPUTS` |
| KG region nodes | `KG_FILE_PATTERNS["regions"]` | `<book>_nodes_regions.json` | If `kg_v3` is active |
| KG edges | `KG_FILE_PATTERNS["edges"]` | `<book>_edges.json` | Recommended: supplies dates and parent regions |
| KG entries, stays, mentions | `KG_FILE_PATTERNS["entries" / "stays" / "mentions"]` | `<book>_nodes_entries.json`, `<book>_nodes_stays.json`, `<book>_mentions.json` | Optional: improve date coverage |
| Place mentions | `PLACE_MENTIONS_CSV` | `place_mentions.csv` from the place mention pipeline | If `place_mentions` is active |
| CGNDB | `CGNDB_DIR` / `CGNDB_GLOB` | Canadian Geographical Names Database CSV, either the national file or the provincial files, UTF-8 | For the `cgndb` gazetteer |
| CHGIS | `CHGIS_PATH` | Canadian Historical GIS census subdivision geodatabase (`.gdb`), or any OGR-readable file with one layer per census year | For the `chgis_name` gazetteer and the `chgis_csd` enricher |
| Overrides | `OVERRIDES_CSV` | CSV, schema below | Optional |
| Generic CSV gazetteers | `GENERIC_CSV_PATHS` | Delimited text, columns mapped in `GENERIC_CSV_SPECS` | Optional |

A gazetteer whose file is missing is skipped with a notice; the remaining routes still run. The notebook stops if no gazetteer is available.

Assumptions about inputs:
- Knowledge-graph files follow the v3 pipeline output: node records carry `location_id` or `region_id`, `canonical_label`, `aliases`, `first_mentioned_entry`; edge records carry `edge_type`, `entry_id`, `from_id`, `to_id`, `from_type`; entry records carry `entry_id` and `entry_date`; stay records carry `stay_id` and `arrival_date`.
- The book code is the file name prefix before `_nodes_`, `_edges` or `_mentions`.
- Dates are ISO 8601 at day, month or year precision. Only the year is used.
- Books with no usable dates take their year from `DEFAULT_YEARS`.
- `place_mentions.csv` contains at least `book`, `entry_id`, `date`, `place`.
- The CGNDB CSV uses the headers listed in the CGN product specification (edition 2.2, Appendix A). Headers are matched after lower-casing and removing punctuation. Required: geographical name, latitude, longitude.
- CHGIS layer and field names are detected from the file (section 3.4) and can be pinned in the configuration cell.

Overrides CSV schema (header row required; `name`, `latitude` and `longitude` are required, other columns optional):

| Column | Description |
|---|---|
| `name` | Place name as it appears in the query (matched after folding) |
| `latitude`, `longitude` | WGS84. A representative point, not a boundary |
| `matched_name` | Label reported in the output |
| `kind` | `location` or `region`; blank applies to both |
| `book` | Book code; blank applies to all books |
| `year_from`, `year_to` | Entry-year range for which the override applies; blank is open-ended |
| `province` | Modern province or territory, if known |
| `feature_type`, `feature_class` | Free text type, and one of the classes in section 3.1 (default `other`) |
| `note` | Free text, not used |

### Outputs

Written to `OUTPUT_DIR`.

| File | Content |
|---|---|
| `place_matches.csv` | Main result. One row per place query (a place within a book). |
| `place_matches.json` | Same records as JSON, keyed by `source_id`, for joining onto knowledge-graph nodes. |
| `place_candidates.csv` | Top-ranked candidates considered for each query, for review. |
| `place_matches.geojson` | Point features for rows with coordinates (`matched` and `inherited`). |
| `csd_polygons.geojson` | Polygons of the census subdivisions that were linked at least once. |
| `run_metadata.json` | Configuration summary, gazetteer manifest and counts. |

Schema of `place_matches.csv`:

| Column | Type | Description |
|---|---|---|
| `query_id` | str | `<source>:<source_id>:<book>`; unique |
| `source` | str | Input adapter: `kg_v3` or `place_mentions` |
| `source_id` | str | Node id in the source (`location_id`, `region_id`), or the folded place string |
| `book` | str | Book code |
| `name` | str | Query name |
| `aliases` | str | Alternative names, `\|`-separated |
| `kind` | str | `location`, `region`, or null if the source does not distinguish |
| `year_min`, `year_max` | int | Range of entry years in which the place occurs |
| `year_source` | str | `entries`, `default_year` or `unknown` |
| `n_mentions` | int | Mention count (`place_mentions`) or entry count (`kg_v3`) |
| `parent_names` | str | Parent region names from `within` edges, `\|`-separated |
| `status` | str | `matched`, `inherited`, `unmatched` |
| `gazetteer` | str | Gazetteer that supplied the match |
| `time_basis` | str | `modern`, `census_<year>`, `override` or the generic source's `time_basis` |
| `feature_id` | str | Gazetteer feature id (CGNDB ID; CSD id; override row) |
| `feature_id_alt` | str | CGNDB Toponymic Feature ID |
| `matched_name` | str | Gazetteer name of the matched feature |
| `matched_via` | str | `canonical`, `alias` or `variant` (name used to match) |
| `feature_type` | str | Gazetteer feature type (CGNDB concise code) |
| `feature_class` | str | Normalised class (section 3.1) |
| `province` | str | Modern province or territory, lower-case, if known |
| `latitude`, `longitude` | float | WGS84 |
| `geometry_type` | str | `point`, `polygon_point` (representative point of a polygon) or `inherited` |
| `name_score` | float | Name similarity after penalties (0–1) |
| `score` | float | Ranking score: name score plus bonuses |
| `n_candidates` | int | Candidates from the winning gazetteer that met the name threshold |
| `ambiguous` | bool | A distinct, distant candidate scored within `AMBIGUITY_MARGIN` |
| `runner_up_name`, `runner_up_distance_km` | str, float | The competing candidate |
| `province_hints` | str | Hints applied to scoring, `\|`-separated |
| `province_hint_met` | bool | Null if no hints; otherwise whether the match lies in a hinted province |
| `inherited_from` | str | Parent `source_id` whose coordinates were inherited |
| `csd_status` | str | `matched`, `not_applicable`, `out_of_period`, `no_year`, `no_coordinates`, `no_polygon` |
| `csd_method` | str | `name`, `point_in_polygon` or `nearest` |
| `csd_layer_year` | int | Census year of the layer used |
| `csd_gap_years` | int | Difference between the place's year and the layer year |
| `csd_id`, `csd_name` | str | Census subdivision |
| `csd_snap_m` | float | Distance to the polygon when `csd_method` is `nearest` |
| `csd_<field>` | str | One column per field in `CHGIS_FIELDS["extra"]` |
| `needs_review` | bool | True if `review_reasons` is non-empty |
| `review_reasons` | str | `\|`-separated: `unmatched`, `ambiguous`, `fuzzy_name`, `alias_match`, `variant_match`, `province_hint_unmet`, `year_unknown`, `inherited`, `csd_snapped`, `csd_no_polygon` |

### Known limitations
- CGNDB records current official names and modern positions. A match is a modern feature with the same name, not a verified historical place.
- Story locations (farms, houses, ships) are not in any gazetteer. They remain `unmatched`, or `inherited` from a parent region if `INHERIT_FROM_PARENT` is set.
- Census subdivision linkage uses modern coordinates against historical polygons. It is reliable for settlements that have not moved and unreliable for features whose position is a representative point of a large area.
- CHGIS covers census years 1851–1921. Books outside that range, with the tolerance in `CHGIS_MAX_GAP_YEARS`, receive `csd_status = out_of_period`.
- The CGNDB carries a content advisory: it contains historical terminology considered racist, offensive or derogatory. Names are matched and reproduced as published.

### Paths

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Every file system location used by the notebook is defined here.
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

# Inputs: knowledge-graph extraction output (v3 pipeline)
KG_DIR          = PROJECT_ROOT / "output" / "v3"
KG_REGISTRY_DIR = KG_DIR / "registries"
KG_FILE_PATTERNS = {                      # (directory, glob) per file type; one file per book
    "locations": (KG_REGISTRY_DIR, "*_nodes_locations.json"),
    "regions":   (KG_REGISTRY_DIR, "*_nodes_regions.json"),
    "entries":   (KG_DIR,          "*_nodes_entries.json"),
    "stays":     (KG_DIR,          "*_nodes_stays.json"),
    "edges":     (KG_DIR,          "*_edges.json"),
    "mentions":  (KG_DIR,          "*_mentions.json"),
}

# Inputs: place mention pipeline output
PLACE_MENTIONS_CSV = PROJECT_ROOT / "output" / "place_mentions.csv"

# Gazetteers
GAZETTEER_DIR     = PROJECT_ROOT / "gazetteers"
CGNDB_DIR         = GAZETTEER_DIR / "cgndb"          # national or provincial CGNDB CSV files
CGNDB_GLOB        = "*.csv"
CHGIS_PATH        = GAZETTEER_DIR / "chgis" / "TCP_CANADA_CSD_202306.gdb"
OVERRIDES_CSV     = GAZETTEER_DIR / "place_overrides.csv"
GENERIC_CSV_PATHS = {}                               # e.g. {"geonames": GAZETTEER_DIR / "geonames" / "CA.txt"}

# Outputs
OUTPUT_DIR        = PROJECT_ROOT / "output" / "place_matching"
MATCHES_CSV       = OUTPUT_DIR / "place_matches.csv"
MATCHES_JSON      = OUTPUT_DIR / "place_matches.json"
CANDIDATES_CSV    = OUTPUT_DIR / "place_candidates.csv"
MATCHES_GEOJSON   = OUTPUT_DIR / "place_matches.geojson"
CSD_GEOJSON       = OUTPUT_DIR / "csd_polygons.geojson"
RUN_METADATA_JSON = OUTPUT_DIR / "run_metadata.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [("KG folder", KG_DIR), ("Place mentions", PLACE_MENTIONS_CSV), ("CGNDB folder", CGNDB_DIR),
                 ("CHGIS", CHGIS_PATH), ("Overrides", OVERRIDES_CSV)]:
    print(f"{label:15}: {p}  {'(exists)' if p.exists() else '(NOT FOUND)'}")
print(f"{'Output folder':15}: {OUTPUT_DIR}")

### Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Inputs
ACTIVE_INPUTS = ["kg_v3"]          # any of: "kg_v3", "place_mentions"
BOOKS         = None               # None = all books; or a list of codes, e.g. ["SofS"]

# Year used for books whose entries carry no usable date (same table as the place mention pipeline)
DEFAULT_YEARS = {
    "AinUTL": 1666, "AToBD": 1862, "BFfH": 1916, "BUoL": 1885,
    "CofOO": 1866, "DofmyC": 1759, "DofTT": 1887, "DRtoF": 1863,
    "EftW": 1940, "FitS": 1815, "NbutC": 1783, "NSH": 1917,
    "OA": 1922, "OatmyD": 1897, "PasWide": 1926, "PintPL": 1914,
    "SofS": 1847, "TA": 1941, "TFN": 1912, "WofW": 1812,
}

# CHGIS (declared before ROUTES, which depends on it)
CHGIS_MAX_GAP_YEARS = 10           # maximum distance between a place's year and the census layer used
CHGIS_CENSUS_RANGE  = (1851, 1921)

# Routing: gazetteers are tried in this order; the first that yields an acceptable match wins.
# Route keys: gazetteer (required); kinds; books; year_from; year_to; min_name_score; require_year
ROUTES = [
    dict(gazetteer="overrides"),
    dict(gazetteer="cgndb"),
    dict(gazetteer="chgis_name", require_year=True,
         year_from=CHGIS_CENSUS_RANGE[0] - CHGIS_MAX_GAP_YEARS,
         year_to=CHGIS_CENSUS_RANGE[1] + CHGIS_MAX_GAP_YEARS),
]
ENRICHERS = ["chgis_csd"]          # applied to matched rows after resolution

# Name matching
MIN_NAME_SCORE   = 0.85            # minimum name similarity to accept a candidate
FUZZY_ENABLED    = True
FUZZY_MIN_LEN    = 4               # names shorter than this are matched exactly only
USE_ALIASES      = True
ALIAS_PENALTY    = 0.05            # subtracted from the name score of a match made through an alias
ARTICLE_PENALTY  = 0.02            # match made after removing a leading "the"/"a"/"an"
VARIANT_PENALTY  = 0.04            # match made after a rule in VARIANT_LEADING / VARIANT_TRAILING
VARIANT_LEADING  = ["city of", "town of", "village of"]
VARIANT_TRAILING = ["city", "town", "village"]
MAX_RAW_CANDIDATES = 200           # per gazetteer and query, before scoring

# Scoring (added to the name score)
KIND_CLASS_BONUS = {               # by query kind, then feature class
    "region":   {"admin_large": 0.10, "area": 0.08, "settlement": 0.05},
    "location": {"settlement": 0.05, "facility": 0.03, "island": 0.03},
}
RELEVANCE_WEIGHT       = 0.05      # weight of CGNDB "relevance at scale" (log-scaled, 0-1)
PROVINCE_MATCH_BONUS   = 0.15
AMBIGUITY_MARGIN       = 0.10      # competitor within this score margin ...
AMBIGUITY_KM           = 10.0      # ... and farther than this distance marks a match ambiguous
MAX_CANDIDATES_LOGGED  = 5         # per query and gazetteer, in place_candidates.csv

# Context hints
BOOK_PROVINCE_HINTS = {}           # e.g. {"SofS": ["quebec"]}; modern province names, lower-case
REGION_NAME_HINTS = {              # historical region name (folded) -> modern province(s); draft
    "canada east":      ["quebec"],
    "lower canada":     ["quebec"],
    "canada west":      ["ontario"],
    "upper canada":     ["ontario"],
    "province of canada": ["quebec", "ontario"],
    "acadia":           ["nova scotia", "new brunswick"],
}
INHERIT_FROM_PARENT = False        # unmatched places take the coordinates of a matched parent region

# Province normalisation (CGNDB values are names; SGC codes and abbreviations are also accepted)
PROVINCE_CODES = {
    "10": "newfoundland and labrador", "11": "prince edward island", "12": "nova scotia",
    "13": "new brunswick", "24": "quebec", "35": "ontario", "46": "manitoba",
    "47": "saskatchewan", "48": "alberta", "59": "british columbia", "60": "yukon",
    "61": "northwest territories", "62": "nunavut",
}
PROVINCE_ALIASES = {
    "nl": "newfoundland and labrador", "newfoundland": "newfoundland and labrador",
    "pe": "prince edward island", "pei": "prince edward island",
    "ns": "nova scotia", "nb": "new brunswick",
    "qc": "quebec", "que": "quebec", "pq": "quebec",
    "on": "ontario", "ont": "ontario", "mb": "manitoba", "man": "manitoba",
    "sk": "saskatchewan", "sask": "saskatchewan", "ab": "alberta", "alta": "alberta",
    "bc": "british columbia", "yt": "yukon", "yuk": "yukon",
    "nt": "northwest territories", "nwt": "northwest territories", "nu": "nunavut",
}

# CGNDB: column headers (matched after lower-casing and removing punctuation) and feature classes
CGNDB_COLUMN_ALIASES = {
    "cgndb_id":   ["cgndbid"],
    "name":       ["geographicalname", "geoname"],
    "generic":    ["genericterm", "generic"],
    "category":   ["genericcategory", "category"],
    "concise":    ["concisecode", "concise"],
    "feature_id": ["toponymicfeatureid", "featureid"],
    "lat":        ["latitude", "lat"],
    "lon":        ["longitude", "lon", "long"],
    "location":   ["location"],
    "province":   ["provinceterritory", "provterr", "province"],
    "relevance":  ["relevanceatscale", "relscale"],
}
CONCISE_CLASS = {                  # CGNDB concise code -> feature class
    "PROV": "admin_large", "TERR": "admin_large",
    "CITY": "settlement", "TOWN": "settlement", "VILG": "settlement", "HAM": "settlement", "DMUN": "settlement",
    "MUN1": "area", "GEOG": "area",
    "RIV": "water", "RIVF": "water", "LAKE": "water", "BAY": "water", "CHAN": "water", "SEAF": "water", "SEAU": "water",
    "ISL": "island", "MTN": "terrain", "CAPE": "terrain",
    "AIR": "facility", "MAR": "facility", "RAIL": "facility", "ROAD": "facility", "CAMP": "facility",
}
CATEGORY_CLASS = {                 # CGNDB generic category (normalised) -> feature class, used if the concise code is unknown
    "populatedplace": "settlement", "administrativearea": "area", "waterfeature": "water",
    "terrainfeature": "terrain", "constructedfeature": "facility",
    "undersea and maritimefeature": "water", "undersea&maritimefeature": "water",
}

# CHGIS: layer discovery, field names and linkage
CHGIS_LAYER_INCLUDE = None         # substring a layer name must contain (e.g. "CSD"); None = no filter
CHGIS_FIELDS = {                   # pin field names here if auto-detection fails; None = auto-detect
    "id": None, "name": None,
    "extra": [],                   # additional fields carried to the output as csd_<field>
}
CHGIS_FIELD_CANDIDATES = {
    "id":   ["ccriuid", "csduid", "csdid", "csd_id", "uid", "id", "geouid"],
    "name": ["csdname", "csd_name", "name", "csd", "nom", "csdnom"],
}
CSD_SNAP_M = 3000                  # a point outside every polygon is assigned the nearest one within this distance (m)
CSD_ELIGIBLE_CLASSES = {"settlement", "area", "island", "terrain", "facility", "csd", "other"}
CSD_GEOJSON_SIMPLIFY_M = None      # simplification tolerance in metres for csd_polygons.geojson; None = none

# Generic CSV gazetteers, one spec per key of GENERIC_CSV_PATHS. Example (GeoNames country dump):
# GENERIC_CSV_SPECS = {"geonames": dict(
#     read_kwargs=dict(sep="\t", header=None, quoting=3, encoding="utf-8",
#                      names=["id","name","ascii","alt","lat","lon","fclass","fcode","cc","cc2","a1","a2","a3","a4","pop","elev","dem","tz","mod"]),
#     name_col="name", lat_col="lat", lon_col="lon", id_col="id", type_col="fcode",
#     province_col=None, alt_names_col="alt", alt_sep=",", time_basis="modern", feature_class="other")}
GENERIC_CSV_SPECS = {}

### Imports

In [ ]:
# ── Imports and dependency check ──────────────────────────────────────────────
import re, json, math, bisect, difflib, unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs):
        return it

print("pandas", pd.__version__, "| geopandas", gpd.__version__, "| shapely", shapely.__version__, "| pyogrio", pyogrio.__version__)

## 1. Place queries

Every input source is converted to one common record, a *place query*: a place name within a book, with the year range in which it occurs and optional context. Matching operates on queries only and does not depend on the source.

| Field | Description |
|---|---|
| `query_id`, `source`, `source_id`, `book` | Identity. One query per place per book |
| `name`, `aliases` | Name to match and alternative names |
| `kind` | `location`, `region` or `None` |
| `year_min`, `year_max`, `year_source` | Years of the entries in which the place occurs |
| `n_mentions` | Mention or entry count |
| `parent_ids`, `parent_names` | Parent regions (`within` edges) |

Adding a source: write a function returning a list of query dicts with these fields and register it in `INPUT_ADAPTERS`.

In [ ]:
# ── Name folding ──────────────────────────────────────────────────────────────
def fold(s) -> str:
    """Normalise a place string for comparison: accents, case, punctuation, Saint/St."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = s.replace("'", "").replace("’", "").replace("‘", "").replace(".", "")
    s = re.sub(r"[-‐‑–—/]", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\bsaint\b", "st", s)
    s = re.sub(r"\bsainte\b", "ste", s)
    return " ".join(s.split())

def year_of(date_str):
    """Four-digit year at the start of an ISO date string, or None."""
    if isinstance(date_str, str) and re.match(r"^\d{4}", date_str):
        return int(date_str[:4])
    return None

def norm_province(x):
    """Lower-case modern province or territory name, or None."""
    if x is None:
        return None
    s = fold(str(x).split("/")[0])
    if not s:
        return None
    return PROVINCE_CODES.get(s) or PROVINCE_ALIASES.get(s) or s

def haversine_km(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * math.asin(math.sqrt(a))

QUERY_FIELDS = ["query_id", "source", "source_id", "book", "name", "aliases", "kind", "year_min", "year_max",
                "year_source", "n_mentions", "parent_ids", "parent_names"]

def _year_fields(book, years):
    if years:
        return min(years), max(years), "entries"
    if DEFAULT_YEARS.get(book):
        return DEFAULT_YEARS[book], DEFAULT_YEARS[book], "default_year"
    return None, None, "unknown"

### 1.1 Knowledge-graph adapter (`kg_v3`)

Reads the location and region node files of each book. For every node, the years of its entries are collected from edge `entry_id` values, the mentions file, stay arrival dates (`at` edges) and `first_mentioned_entry`. Parent regions come from `within` edges.

> **[DESIGN NOTE — delete before publishing]**
> **Years are derived from edges, not from the node record.** The v3 node files carry no dates; dates exist on entries and stays. Every edge stores the `entry_id` it was extracted from, so following `edge.entry_id → entry.entry_date` gives a date for any node that appears in an edge, without a graph traversal. Stay arrival dates are added because a stay's narrated arrival can differ from its entry date. The result is a year *range* per place per book, which is what time-based routing needs.
>
> **One query per place per book.** A place that occurs in two books is resolved twice, with that book's years and hints. Resolution is cheap because gazetteer lookups are in memory, and it keeps the time routing correct for places such as Quebec City that recur across periods.
>
> **Nodes are not merged across books.** Cross-book entity merging is a separate step. Matching to a shared gazetteer feature id is one way to perform it, using the `feature_id` column of the output.

In [ ]:
def _read_kg_files(kind):
    directory, pattern = KG_FILE_PATTERNS[kind]
    out = {}
    for f in sorted(Path(directory).glob(pattern)):
        book = re.split(r"_nodes_|_edges|_mentions", f.name)[0]
        out[book] = json.loads(f.read_text(encoding="utf-8"))
    return out

def load_kg_v3(books=None):
    locations, regions = _read_kg_files("locations"), _read_kg_files("regions")
    if not locations and not regions:
        raise FileNotFoundError(f"No KG location or region files found under {KG_REGISTRY_DIR}")
    entries, stays = _read_kg_files("entries"), _read_kg_files("stays")
    edges, mentions = _read_kg_files("edges"), _read_kg_files("mentions")

    queries = []
    for book in sorted(set(locations) | set(regions)):
        if books is not None and book not in books:
            continue
        entry_date = {e["entry_id"]: e.get("entry_date") for e in entries.get(book, [])}
        stay_arrival = {s["stay_id"]: s.get("arrival_date") for s in stays.get(book, [])}
        node_entries, node_dates, parents = defaultdict(set), defaultdict(list), defaultdict(list)
        for e in edges.get(book, []):
            if e.get("entry_id"):
                for end in (e.get("from_id"), e.get("to_id")):
                    if end:
                        node_entries[end].add(e["entry_id"])
            if e.get("edge_type") == "at" and e.get("from_type") == "stay" and stay_arrival.get(e["from_id"]):
                node_dates[e["to_id"]].append(stay_arrival[e["from_id"]])
            if e.get("edge_type") == "within":
                parents[e["from_id"]].append(e["to_id"])
        mention_count = Counter()
        for m in mentions.get(book, []):
            if m.get("entity_id") and m.get("entry_id"):
                node_entries[m["entity_id"]].add(m["entry_id"])
                mention_count[m["entity_id"]] += 1
        region_label = {r["region_id"]: r.get("canonical_label") for r in regions.get(book, [])}

        for kind, nodes, id_field in (("region", regions.get(book, []), "region_id"),
                                      ("location", locations.get(book, []), "location_id")):
            for n in nodes:
                nid, label = n[id_field], n.get("canonical_label") or n[id_field]
                entry_ids = set(node_entries.get(nid, ()))
                if n.get("first_mentioned_entry"):
                    entry_ids.add(n["first_mentioned_entry"])
                dates = [entry_date.get(i) for i in entry_ids] + node_dates.get(nid, [])
                y_min, y_max, y_src = _year_fields(book, [y for y in map(year_of, dates) if y is not None])
                parent_ids = [p for p in parents.get(nid, []) if p in region_label]
                queries.append(dict(
                    query_id=f"kg_v3:{nid}:{book}", source="kg_v3", source_id=nid, book=book, name=label,
                    aliases=[a for a in n.get("aliases", []) if a and fold(a) != fold(label)],
                    kind=kind, year_min=y_min, year_max=y_max, year_source=y_src,
                    n_mentions=mention_count.get(nid) or len(entry_ids),
                    parent_ids=parent_ids, parent_names=[region_label[p] for p in parent_ids]))
    return queries

### 1.2 Place mention adapter (`place_mentions`)

Reads `place_mentions.csv` and groups mentions by book and folded place string. `kind` is null because NER labels do not distinguish locations from regions.

In [ ]:
def load_place_mentions(books=None):
    df = pd.read_csv(PLACE_MENTIONS_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
    missing = [c for c in ("book", "entry_id", "date", "place") if c not in df.columns]
    if missing:
        raise ValueError(f"{PLACE_MENTIONS_CSV.name}: missing columns {missing}")
    if books is not None:
        df = df[df["book"].isin(books)]
    df = df.assign(place_key=df["place"].map(fold))
    df = df[df["place_key"] != ""]

    queries = []
    for (book, key), g in df.groupby(["book", "place_key"], sort=True):
        y_min, y_max, y_src = _year_fields(book, [y for y in map(year_of, g["date"]) if y is not None])
        queries.append(dict(
            query_id=f"place_mentions:{key}:{book}", source="place_mentions", source_id=key, book=book,
            name=g["place"].value_counts().index[0], aliases=[], kind=None,
            year_min=y_min, year_max=y_max, year_source=y_src, n_mentions=len(g),
            parent_ids=[], parent_names=[]))
    return queries

### 1.3 Combine inputs

In [ ]:
INPUT_ADAPTERS = {"kg_v3": load_kg_v3, "place_mentions": load_place_mentions}

def build_queries():
    queries = []
    for name in ACTIVE_INPUTS:
        got = INPUT_ADAPTERS[name](BOOKS)
        print(f"{name}: {len(got)} queries")
        queries.extend(got)
    ids = Counter(q["query_id"] for q in queries)
    dupes = [k for k, v in ids.items() if v > 1]
    if dupes:
        raise ValueError(f"Duplicate query_id values: {dupes[:5]}")
    if not queries:
        raise ValueError("No place queries were produced; check ACTIVE_INPUTS and the input paths.")
    return queries

QUERIES = build_queries()
pd.DataFrame(QUERIES).groupby(["source", "book", "kind"], dropna=False).agg(
    queries=("query_id", "size"), year_min=("year_min", "min"), year_max=("year_max", "max")).reset_index()

## 2. Name matching

Each query is expanded into name *variants*: the canonical name, its aliases, and normalised forms. Every variant carries a penalty that is subtracted from the name score of any match made through it. Gazetteers index folded names and are searched with the variants.

Lookup is exact on the folded name. If no variant has an exact hit, a fuzzy search over similar names is performed (similarity ratio at or above `MIN_NAME_SCORE`).

> **[DESIGN NOTE — delete before publishing]**
> **Fuzzy search is blocked by first character and length.** Comparing a query with every name in the national CGNDB (several hundred thousand) using `difflib` is too slow to repeat for every query. Names are grouped by first character and sorted by length; only names whose length can still reach the threshold are compared, after the cheap `real_quick_ratio` and `quick_ratio` upper bounds. Cost: a name with a misspelt *first* letter is never fuzzy-matched. Misspellings elsewhere are found. `difflib` is used instead of a faster package so that scores do not depend on which packages are installed.
>
> **Variant penalties are small and additive.** "The Pas" is a place name, so a leading article is removed only as a fallback variant, with a small penalty so that the unmodified name wins when both match. The same applies to "Quebec City" → "Quebec". The rule lists (`VARIANT_LEADING`, `VARIANT_TRAILING`) are short on purpose: every rule can create false matches ("Mount Royal City"). Extend them from the review output rather than in advance.
>
> **Alias matches are penalised, not excluded.** Aliases are surface forms collected by the extraction step ("the island", "the sheds") and are less reliable than canonical labels. Set `USE_ALIASES = False` to ignore them.

In [ ]:
@dataclass
class Variant:
    text: str       # folded
    via: str        # canonical | alias | variant
    penalty: float

_LEADING_ARTICLE = re.compile(r"^(?:the|a|an)\s+")

def build_variants(query):
    sources = [(query["name"], "canonical", 0.0)]
    if USE_ALIASES:
        sources += [(a, "alias", ALIAS_PENALTY) for a in query["aliases"]]
    out = {}

    def add(text, via, penalty):
        if text and (text not in out or penalty < out[text].penalty):
            out[text] = Variant(text, via, penalty)

    for surface, via, base in sources:
        f = fold(surface)
        add(f, via, base)
        derived = "alias" if via == "alias" else "variant"
        stripped = _LEADING_ARTICLE.sub("", f)
        if stripped != f:
            add(stripped, derived, base + ARTICLE_PENALTY)
        for lead in VARIANT_LEADING:
            if stripped.startswith(lead + " "):
                add(stripped[len(lead) + 1:], derived, base + VARIANT_PENALTY)
        for trail in VARIANT_TRAILING:
            if stripped.endswith(" " + trail):
                add(stripped[:-len(trail) - 1], derived, base + VARIANT_PENALTY)
    return list(out.values())


class NameIndex:
    """Exact and fuzzy lookup over folded names. `pairs` is an iterable of (folded_name, row_position)."""

    def __init__(self, pairs):
        self.exact = defaultdict(list)
        for name, pos in pairs:
            if name:
                self.exact[name].append(pos)
        blocks = defaultdict(list)
        for name in self.exact:
            blocks[name[0]].append((len(name), name))
        self._blocks = {k: sorted(v) for k, v in blocks.items()}
        self._lens = {k: [n for n, _ in v] for k, v in self._blocks.items()}

    def fuzzy(self, q, cutoff, limit=25):
        if len(q) < FUZZY_MIN_LEN or q[0] not in self._blocks:
            return []
        lens, block = self._lens[q[0]], self._blocks[q[0]]
        lo = int(len(q) * cutoff / (2 - cutoff))
        hi = int(len(q) * (2 - cutoff) / cutoff) + 1
        sm = difflib.SequenceMatcher(autojunk=False)
        sm.set_seq2(q)
        hits = []
        for _, cand in block[bisect.bisect_left(lens, lo):bisect.bisect_right(lens, hi)]:
            sm.set_seq1(cand)
            if sm.real_quick_ratio() < cutoff or sm.quick_ratio() < cutoff:
                continue
            r = sm.ratio()
            if r >= cutoff:
                hits.append((r, cand))
        hits.sort(reverse=True)
        return [(c, r) for r, c in hits[:limit]]


def name_hits(index, variants):
    """{row_position: (name_score, via)} for the variants of one query."""
    hits, any_exact = {}, False
    for v in variants:
        for pos in index.exact.get(v.text, ()):
            any_exact = True
            s = 1.0 - v.penalty
            if pos not in hits or s > hits[pos][0]:
                hits[pos] = (s, v.via)
    if not any_exact and FUZZY_ENABLED:
        for v in variants:
            for name, ratio in index.fuzzy(v.text, MIN_NAME_SCORE):
                s = ratio - v.penalty
                for pos in index.exact[name]:
                    if pos not in hits or s > hits[pos][0]:
                        hits[pos] = (s, v.via)
    return hits

## 3. Gazetteers

A gazetteer is an object with a `name`, a `time_basis` label and a method `candidates(query, variants)` that returns a list of `Candidate` records. Candidates carry the gazetteer's own feature id, name, coordinates, type and province. Scoring and acceptance are applied outside the gazetteer, so all gazetteers are ranked on the same terms.

Adding a gazetteer: subclass `Gazetteer`, implement `candidates`, instantiate it in `build_gazetteers()` and add a route in `ROUTES`.

In [ ]:
@dataclass
class Candidate:
    gazetteer: str
    time_basis: str
    feature_id: str
    name: str
    lat: float
    lon: float
    feature_id_alt: str = None
    feature_type: str = None
    feature_class: str = "other"
    province: str = None
    relevance: float = None
    geometry_type: str = "point"
    name_score: float = 0.0
    matched_via: str = "canonical"
    score: float = 0.0
    extra: dict = field(default_factory=dict)


class Gazetteer:
    name = "base"
    time_basis = "unspecified"
    n_records = 0
    source = ""

    def candidates(self, query, variants):
        raise NotImplementedError


def _norm_header(h):
    return re.sub(r"[^a-z0-9]", "", str(h).lower())

def detect_columns(headers, aliases, required):
    norm = {_norm_header(h): h for h in headers}
    found = {}
    for key, options in aliases.items():
        for o in options:
            if o in norm:
                found[key] = norm[o]
                break
    missing = [k for k in required if k not in found]
    if missing:
        raise ValueError(f"Could not find columns for {missing}. Headers present: {list(headers)}")
    return found

def _to_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

### 3.1 Overrides

Manual entries (schema in section 0). Resolved first, by exact folded name. Intended for names no gazetteer resolves correctly, such as historical regions and places whose modern namesake is the wrong feature.

Feature classes used by the scoring and enrichment rules: `admin_large` (province, territory, country), `area` (named area, district municipality), `settlement`, `water`, `island`, `terrain`, `facility`, `csd` (census subdivision), `other`.

In [ ]:
class OverridesGazetteer(Gazetteer):
    name = "overrides"
    time_basis = "override"

    def __init__(self, path):
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8")
        df.columns = [c.strip().lower() for c in df.columns]
        missing = [c for c in ("name", "latitude", "longitude") if c not in df.columns]
        if missing:
            raise ValueError(f"{Path(path).name}: missing columns {missing}")
        for c in ("matched_name", "kind", "book", "year_from", "year_to", "province", "feature_type", "feature_class"):
            if c not in df.columns:
                df[c] = ""
        self.df = df.reset_index(drop=True)
        self.index = NameIndex((fold(n), i) for i, n in enumerate(self.df["name"]))
        self.n_records, self.source = len(self.df), str(path)

    def candidates(self, query, variants):
        y_lo, y_hi = query["year_min"], query["year_max"]
        out = {}
        for v in variants:
            for i in self.index.exact.get(v.text, ()):
                r = self.df.iloc[i]
                if r["kind"] and query["kind"] and r["kind"] != query["kind"]:
                    continue
                if r["book"] and r["book"] != query["book"]:
                    continue
                f, t = _to_int(r["year_from"]), _to_int(r["year_to"])
                if y_lo is not None and ((f is not None and y_hi < f) or (t is not None and y_lo > t)):
                    continue
                try:
                    lat, lon = float(r["latitude"]), float(r["longitude"])
                except ValueError:
                    continue
                score = 1.0 - v.penalty
                if i not in out or score > out[i].name_score:
                    out[i] = Candidate(
                        gazetteer=self.name, time_basis=self.time_basis, feature_id=f"override:{i + 2}",
                        name=r["matched_name"] or r["name"], lat=lat, lon=lon,
                        feature_type=r["feature_type"] or None, feature_class=r["feature_class"] or "other",
                        province=norm_province(r["province"]), geometry_type="point",
                        name_score=score, matched_via=v.via)
        return list(out.values())

### 3.2 CGNDB

Canadian Geographical Names Database, read from the downloaded CSV files (national file, or one file per province; records are de-duplicated on id, name and province). One record per name; names of the same feature share a Toponymic Feature ID and are collapsed to the best-scoring name when candidates are built.

Feature class is taken from the concise code (`CONCISE_CLASS`), falling back to the generic category (`CATEGORY_CLASS`). Relevance at scale is read as a number and used as a small prominence bonus in scoring.

> **[DESIGN NOTE — delete before publishing]**
> **Offline CSV instead of the GeoNames search API.** The CGNDB offers both. The CSV was chosen because matching sends hundreds of queries, each possibly fuzzy; offline files make the run deterministic, free of rate limits and reproducible from a dated download (the files are regenerated weekly, so record the download date). The API returns at most 1,000 records per request and has no documented fuzzy search. The cost is that the CSV contains only *current official* names. Formerly official names are available only through the API (`category=H`), so a historical name that was later replaced will not be found here. Use an overrides entry or a second gazetteer for those.
>
> **Column detection is by normalised header, not by position.** The CGN product specification (edition 2.2) lists the CSV attribute names, but the files were not available when this notebook was written, so exact header strings and the content of `Province - Territory` (name or code) could not be verified. Headers are matched after lower-casing and removing punctuation, province values are normalised from names, SGC codes or abbreviations, and the detected mapping is printed on load. If the mapping is wrong, edit `CGNDB_COLUMN_ALIASES`.
>
> **Concise-code classes cover only the codes seen in the specification and API output.** Unknown codes fall back to the generic category, then to `other`. Extend `CONCISE_CLASS` as the review output shows misclassifications; the classes only influence the scoring bonus and whether a census subdivision is attempted.

In [ ]:
def classify_cgndb(concise, category):
    c = CONCISE_CLASS.get(str(concise).strip().upper())
    return c or CATEGORY_CLASS.get(_norm_header(category), "other")


class CGNDBGazetteer(Gazetteer):
    name = "cgndb"
    time_basis = "modern"

    def __init__(self, directory, pattern):
        files = sorted(Path(directory).glob(pattern))
        if not files:
            raise FileNotFoundError(f"No files matching {pattern} in {directory}")
        frames, self.columns_detected = [], {}
        for f in files:
            try:
                raw = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8")
            except UnicodeDecodeError:
                raw = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="latin-1")
            cols = detect_columns(raw.columns, CGNDB_COLUMN_ALIASES, required=["name", "lat", "lon"])
            self.columns_detected = cols
            sub = pd.DataFrame({k: raw[v] for k, v in cols.items()})
            for k in CGNDB_COLUMN_ALIASES:
                if k not in sub:
                    sub[k] = ""
            frames.append(sub)
        df = pd.concat(frames, ignore_index=True)
        if (df["cgndb_id"] != "").any():
            df = df.drop_duplicates(["cgndb_id", "name", "province"])
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
        df["relevance"] = pd.to_numeric(df["relevance"], errors="coerce")
        df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

        self._id = df["cgndb_id"].tolist()
        self._uid = df["feature_id"].tolist()
        self._name = df["name"].tolist()
        self._lat, self._lon = df["lat"].tolist(), df["lon"].tolist()
        self._concise = df["concise"].tolist()
        self._rel = df["relevance"].tolist()
        self._province = [norm_province(p) for p in df["province"]]
        self._class = [classify_cgndb(c, g) for c, g in zip(df["concise"], df["category"])]
        self.index = NameIndex(zip(df["name"].map(fold), range(len(df))))
        self.n_records = len(df)
        self.source = f"{len(files)} file(s) in {directory}"

    def candidates(self, query, variants):
        hits = name_hits(self.index, variants)
        order = sorted(hits, key=lambda p: (-hits[p][0], -(self._rel[p] if self._rel[p] == self._rel[p] else 0)))
        out = {}
        for p in order[:MAX_RAW_CANDIDATES]:
            score, via = hits[p]
            key = self._uid[p] or self._id[p] or p
            if key in out and out[key].name_score >= score:
                continue
            rel = self._rel[p]
            out[key] = Candidate(
                gazetteer=self.name, time_basis=self.time_basis, feature_id=self._id[p], feature_id_alt=self._uid[p] or None,
                name=self._name[p], lat=self._lat[p], lon=self._lon[p], feature_type=self._concise[p] or None,
                feature_class=self._class[p], province=self._province[p],
                relevance=None if rel != rel else rel, geometry_type="point", name_score=score, matched_via=via)
        return list(out.values())

### 3.3 Generic CSV gazetteer

Any delimited file with a name column and coordinates. The column mapping is given per source in `GENERIC_CSV_SPECS`. Alternate names (`alt_names_col`) are indexed as the gazetteer's own names. Use this to add GeoNames, Natural Earth or a hand-built list for a period or region not covered by the other gazetteers.

In [ ]:
class GenericCSVGazetteer(Gazetteer):
    def __init__(self, name, path, spec):
        self.name = name
        self.time_basis = spec.get("time_basis", "unspecified")
        self._class_default = spec.get("feature_class", "other")
        df = pd.read_csv(path, dtype=str, keep_default_na=False, **spec.get("read_kwargs", {}))
        for key in ("name_col", "lat_col", "lon_col"):
            if spec[key] not in df.columns:
                raise ValueError(f"{name}: column {spec[key]!r} not in {Path(path).name}; columns: {list(df.columns)}")
        df["_lat"] = pd.to_numeric(df[spec["lat_col"]], errors="coerce")
        df["_lon"] = pd.to_numeric(df[spec["lon_col"]], errors="coerce")
        df = df.dropna(subset=["_lat", "_lon"]).reset_index(drop=True)
        self._name = df[spec["name_col"]].tolist()
        self._lat, self._lon = df["_lat"].tolist(), df["_lon"].tolist()
        col = lambda k: df[spec[k]].tolist() if spec.get(k) else [None] * len(df)
        self._id, self._type = col("id_col"), col("type_col")
        self._province = [norm_province(p) if p else None for p in col("province_col")]
        pairs = [(fold(n), i) for i, n in enumerate(self._name)]
        if spec.get("alt_names_col"):
            sep = spec.get("alt_sep", ",")
            for i, alts in enumerate(df[spec["alt_names_col"]]):
                pairs += [(fold(a), i) for a in str(alts).split(sep) if a.strip()]
        self.index = NameIndex(pairs)
        self.n_records, self.source = len(df), str(path)

    def candidates(self, query, variants):
        hits = name_hits(self.index, variants)
        out = []
        for p in sorted(hits, key=lambda p: -hits[p][0])[:MAX_RAW_CANDIDATES]:
            score, via = hits[p]
            out.append(Candidate(
                gazetteer=self.name, time_basis=self.time_basis, feature_id=str(self._id[p] if self._id[p] is not None else p),
                name=self._name[p], lat=self._lat[p], lon=self._lon[p], feature_type=self._type[p],
                feature_class=self._class_default, province=self._province[p], geometry_type="point",
                name_score=score, matched_via=via))
        return out

### 3.4 CHGIS

The Canadian Historical GIS provides census subdivision (CSD) polygons for the census years 1851 to 1921, one layer per year. `CHGISLayers` discovers the layers (a four-digit census year in the layer name), detects the id and name fields, and loads layers on demand. It serves two components:

- `chgis_name` (gazetteer): matches a place name against CSD names in the layer nearest to the place's year. The reported point is the representative point of the polygon.
- `chgis_csd` (enricher, section 4.3): assigns a census subdivision to places already matched by another gazetteer.

For a place with year *Y*, the layer used is the census year nearest to *Y* (ties go to the earlier year), provided the distance is at most `CHGIS_MAX_GAP_YEARS`.

> **[DESIGN NOTE — delete before publishing]**
> **Layer choice is the nearest census year, not the latest year not after the entry.** CHGIS polygons re-create the geography *as published for each census*. A diary dated 1847 predates every layer, and one dated 1855 sits between two. The nearest layer minimises the years over which boundaries may have changed; "latest not after" would leave all pre-1851 books unmatched. Ties go to the earlier year. The gap is written to `csd_gap_years` so any threshold can be applied afterwards, and `CHGIS_MAX_GAP_YEARS` bounds it in advance.
>
> **The place's year is the midpoint of its year range.** A place occurring over several years gets one layer. If the range spans several census years, split the book or tighten the range.
>
> **Polygons are reported as a representative point.** `representative_point()` is used instead of the centroid because it is guaranteed to lie inside the polygon; a centroid can fall outside a crescent- or coast-shaped subdivision, which would break the point-in-polygon round trip. The polygon itself is exported separately (`csd_polygons.geojson`).
>
> **Layers are kept in their native projection.** The geodatabase uses a Lambert conformal conic projection in metres. Keeping it makes the snapping tolerance in section 4.3 a distance in metres and avoids reprojecting thousands of polygons. Points are projected into the layer CRS instead.
>
> **Field names are detected, not assumed.** The CHGIS documentation package was not available when this notebook was written, so the id and name field names are unverified. Detection tries `CHGIS_FIELD_CANDIDATES` against the layer's fields. `describe_chgis()` lists layers and fields; pin the names in `CHGIS_FIELDS` if detection picks the wrong ones.

In [ ]:
class CHGISLayers:
    def __init__(self, path):
        if not Path(path).exists():
            raise FileNotFoundError(path)
        self.path = str(path)
        listing = pyogrio.list_layers(self.path)
        self.layer_names, self._cache, self._index_cache, self._tf_cache = {}, {}, {}, {}
        lo, hi = CHGIS_CENSUS_RANGE
        for name, _geom in listing:
            if CHGIS_LAYER_INCLUDE and CHGIS_LAYER_INCLUDE not in name:
                continue
            m = re.search(r"(?<!\d)(1[89]\d{2}|19[0-2]\d)(?!\d)", name)
            if not m or not lo <= int(m.group(1)) <= hi:
                continue
            year = int(m.group(1))
            if year in self.layer_names:
                raise ValueError(f"Layers {self.layer_names[year]!r} and {name!r} both match census year {year}; "
                                 f"set CHGIS_LAYER_INCLUDE to select one kind of layer.")
            self.layer_names[year] = name
        if not self.layer_names:
            raise ValueError(f"No layers with a census year in {Path(path).name}. Layers present: {[n for n, _ in listing]}")

    @property
    def years(self):
        return sorted(self.layer_names)

    def pick(self, year):
        """(layer_year, gap) for the layer nearest to `year`; layer_year is None if the gap exceeds the maximum."""
        if year is None:
            return None, None
        layer_year = min(self.years, key=lambda y: (abs(y - year), y))
        gap = abs(layer_year - year)
        return (layer_year, gap) if gap <= CHGIS_MAX_GAP_YEARS else (None, gap)

    def get(self, year):
        if year not in self._cache:
            self._cache[year] = self._prepare(year)
        return self._cache[year]

    def extra_columns(self):
        return [f"csd_{_norm_header(f)}" for f in CHGIS_FIELDS["extra"]]

    def _prepare(self, year):
        gdf = gpd.read_file(self.path, layer=self.layer_names[year])
        if gdf.crs is None:
            raise ValueError(f"Layer {self.layer_names[year]!r} has no CRS")
        by_norm = {_norm_header(c): c for c in gdf.columns}

        def pick_field(key):
            if CHGIS_FIELDS.get(key):
                if CHGIS_FIELDS[key] not in gdf.columns:
                    raise ValueError(f"CHGIS_FIELDS[{key!r}]={CHGIS_FIELDS[key]!r} not in layer {self.layer_names[year]!r}; "
                                     f"fields: {list(gdf.columns)}")
                return CHGIS_FIELDS[key]
            for cand in CHGIS_FIELD_CANDIDATES[key]:
                if cand in by_norm:
                    return by_norm[cand]
            return None

        id_col, name_col = pick_field("id"), pick_field("name")
        if name_col is None:
            raise ValueError(f"No CSD name field detected in layer {self.layer_names[year]!r}. Fields: {list(gdf.columns)}. "
                             f"Set CHGIS_FIELDS['name'].")
        out = gpd.GeoDataFrame({
            "csd_id": gdf[id_col].astype(str) if id_col else [f"{year}:{i}" for i in range(len(gdf))],
            "csd_name": gdf[name_col].astype(str),
        }, geometry=gdf.geometry, crs=gdf.crs)
        for f, col in zip(CHGIS_FIELDS["extra"], self.extra_columns()):
            out[col] = gdf[f].astype(str) if f in gdf.columns else None
        return out[out.geometry.notna() & ~out.geometry.is_empty].reset_index(drop=True)

    def name_index(self, year):
        if year not in self._index_cache:
            layer = self.get(year)
            pairs = []
            for i, n in enumerate(layer["csd_name"]):
                pairs.append((fold(n), i))
                bare = fold(re.sub(r"\([^)]*\)", "", n))
                if bare != fold(n):
                    pairs.append((bare, i))
            self._index_cache[year] = NameIndex(pairs)
        return self._index_cache[year]

    def to_wgs84(self, year, x, y):
        if year not in self._tf_cache:
            from pyproj import Transformer
            self._tf_cache[year] = Transformer.from_crs(self.get(year).crs, 4326, always_xy=True)
        lon, lat = self._tf_cache[year].transform(x, y)
        return lat, lon


class CHGISGazetteer(Gazetteer):
    name = "chgis_name"

    def __init__(self, layers):
        self.layers = layers
        self.time_basis = "census"
        self.n_records = len(layers.layer_names)
        self.source = f"{layers.path} (layers: {layers.years})"

    def candidates(self, query, variants):
        if query["year_min"] is None:
            return []
        year = round((query["year_min"] + query["year_max"]) / 2)
        layer_year, gap = self.layers.pick(year)
        if layer_year is None:
            return []
        layer, index = self.layers.get(layer_year), self.layers.name_index(layer_year)
        hits = name_hits(index, variants)
        extra_cols = self.layers.extra_columns()
        out = []
        for p in sorted(hits, key=lambda p: -hits[p][0])[:MAX_RAW_CANDIDATES]:
            score, via = hits[p]
            row = layer.iloc[p]
            pt = row.geometry.representative_point()
            lat, lon = self.layers.to_wgs84(layer_year, pt.x, pt.y)
            out.append(Candidate(
                gazetteer=self.name, time_basis=f"census_{layer_year}", feature_id=row["csd_id"], name=row["csd_name"],
                lat=lat, lon=lon, feature_type="census subdivision", feature_class="csd",
                geometry_type="polygon_point", name_score=score, matched_via=via,
                extra=dict(layer_year=layer_year, gap=gap, csd_id=row["csd_id"], csd_name=row["csd_name"],
                           fields={c: row[c] for c in extra_cols})))
        return out


def describe_chgis(layers):
    """List the layers and fields detected in the CHGIS file."""
    rows = []
    for y in layers.years:
        info = pyogrio.read_info(layers.path, layer=layers.layer_names[y])
        rows.append(dict(census_year=y, layer=layers.layer_names[y], features=info["features"],
                         geometry=info["geometry_type"], fields=", ".join(info["fields"])))
    return pd.DataFrame(rows)

### 3.5 Load gazetteers

Instantiates the gazetteers whose files are present and prints a manifest. Routes that reference an unavailable gazetteer are skipped.

In [ ]:
CHGIS = None
GAZETTEERS, GAZETTEER_NOTES = {}, []

def build_gazetteers():
    global CHGIS
    gz, notes = {}, []

    def attempt(label, factory):
        try:
            gz_obj = factory()
            gz[gz_obj.name] = gz_obj
        except FileNotFoundError as e:
            notes.append(f"{label}: not available ({e})")

    attempt("overrides", lambda: OverridesGazetteer(OVERRIDES_CSV))
    attempt("cgndb", lambda: CGNDBGazetteer(CGNDB_DIR, CGNDB_GLOB))
    try:
        CHGIS = CHGISLayers(CHGIS_PATH)
        gz["chgis_name"] = CHGISGazetteer(CHGIS)
    except FileNotFoundError as e:
        CHGIS = None
        notes.append(f"chgis: not available ({e})")
    for key, path in GENERIC_CSV_PATHS.items():
        attempt(key, lambda key=key, path=path: GenericCSVGazetteer(key, path, GENERIC_CSV_SPECS[key]))
    return gz, notes

GAZETTEERS, GAZETTEER_NOTES = build_gazetteers()
for n in GAZETTEER_NOTES:
    print("NOTE", n)
if not GAZETTEERS:
    raise RuntimeError("No gazetteer is available; check the gazetteer paths.")
if "cgndb" in GAZETTEERS:
    print("CGNDB columns detected:", GAZETTEERS["cgndb"].columns_detected)
pd.DataFrame([dict(gazetteer=g.name, time_basis=g.time_basis, records=g.n_records, source=g.source)
              for g in GAZETTEERS.values()])

In [ ]:
# CHGIS layers and fields (run after loading; use to pin CHGIS_FIELDS if detection is wrong)
describe_chgis(CHGIS) if CHGIS is not None else "CHGIS not loaded"

## 4. Matching

### 4.1 Routing, scoring and acceptance

Routes are tried in order. A route applies to a query if the query's kind, book and year range satisfy the route's filters. A query with an unknown year is passed to routes unless `require_year` is set. For each applicable route the gazetteer's candidates are scored and the route is accepted if at least one candidate has `name_score >= MIN_NAME_SCORE`; remaining routes are then skipped.

Score = name score + class bonus (`KIND_CLASS_BONUS`) + relevance bonus (CGNDB only) + province bonus (candidate province among the query's hints). Candidates are ranked by score. The top candidate is the match. A match is *ambiguous* if the highest-ranked candidate that is a different feature and farther than `AMBIGUITY_KM` scores within `AMBIGUITY_MARGIN` of the match.

Province hints for a query come from, in order: the resolved province of its parent regions, `REGION_NAME_HINTS` for parent names (and for its own name if it is a region), and `BOOK_PROVINCE_HINTS`. Regions are resolved before locations so that hints are available.

> **[DESIGN NOTE — delete before publishing]**
> **Cascade instead of scoring all gazetteers together.** Scores from different gazetteers are not comparable: CGNDB offers a modern point for a name, CHGIS a historical polygon. A cascade makes the policy explicit and editable (order of `ROUTES`) and keeps cost low. Cost: a poor first-route match is never compared with a better one later. The `place_candidates.csv` log lists what the first matching route saw; to compare gazetteers directly, run with a single route each and diff the outputs.
>
> **Prior knowledge enters as small additive bonuses, not filters.** A province hint or a place-kind preference changes the ranking among candidates that already match by name, and never excludes one. Weights are guesses that should be tuned against reviewed matches. Province hints are a bonus rather than a constraint because the parent region can itself be wrong, and because the historical region names that supply hints do not map one-to-one onto modern provinces.
>
> **Relevance at scale is a tie-breaker.** Diary writers mention prominent places more often than minor ones; among identically named CGNDB features, the one shown on small-scale maps is more likely to be meant. The weight is deliberately small (0.05) so that it cannot outweigh a province hint or an exact-name difference.
>
> **Ambiguity ignores nearby duplicates.** The CGNDB holds several records for one feature and nearby features with the same name (a river and its mouth). Only competitors farther than `AMBIGUITY_KM` count; same-feature and nearby duplicates do not make a match ambiguous.

In [ ]:
@dataclass
class Route:
    gazetteer: str
    kinds: list = None
    books: list = None
    year_from: int = None
    year_to: int = None
    min_name_score: float = None
    require_year: bool = False

    def applies(self, q):
        if self.kinds and q["kind"] and q["kind"] not in self.kinds:
            return False
        if self.books and q["book"] not in self.books:
            return False
        if q["year_min"] is None:
            return not self.require_year
        if self.year_from is not None and q["year_max"] < self.year_from:
            return False
        if self.year_to is not None and q["year_min"] > self.year_to:
            return False
        return True

ROUTE_OBJECTS = [Route(**r) for r in ROUTES]
_missing = [r.gazetteer for r in ROUTE_OBJECTS if r.gazetteer not in GAZETTEERS]
if _missing:
    print("NOTE: routes skipped, gazetteer not loaded:", sorted(set(_missing)))


def score_candidate(c, kind, hints):
    bonus = KIND_CLASS_BONUS.get(kind, {}).get(c.feature_class, 0.0)
    if c.relevance and c.relevance > 0:
        bonus += RELEVANCE_WEIGHT * min(1.0, math.log10(c.relevance) / math.log10(30_000_000))
    if hints and c.province in hints:
        bonus += PROVINCE_MATCH_BONUS
    c.score = c.name_score + bonus


def resolve_one(q, hints):
    variants = build_variants(q)
    considered, win = [], None
    for route in ROUTE_OBJECTS:
        if route.gazetteer not in GAZETTEERS or not route.applies(q):
            continue
        cands = GAZETTEERS[route.gazetteer].candidates(q, variants)
        for c in cands:
            score_candidate(c, q["kind"], hints)
        cands.sort(key=lambda c: (-c.score, str(c.feature_id)))
        considered.append((route.gazetteer, cands))
        threshold = MIN_NAME_SCORE if route.min_name_score is None else route.min_name_score
        accepted = [c for c in cands if c.name_score >= threshold]
        if accepted:
            win = (route.gazetteer, accepted)
            break
    return win, considered

### 4.2 Hints, resolution loop and parent inheritance

> **[DESIGN NOTE — delete before publishing]**
> **Optional inheritance from the parent region.** Story locations ("Uncle Liam's farm") are in no gazetteer, so most of the locations of a diary remain unmatched. With `INHERIT_FROM_PARENT = True`, an unmatched place whose `within` parent is matched receives the parent's coordinates, with `status = inherited`, `geometry_type = inherited` and `inherited_from` set, so it can be drawn on a map at region precision and filtered out of any analysis that needs real positions. It is off by default because it produces coordinates that are not the place's own. Inherited rows are never given a census subdivision.

In [ ]:
def province_hints(q, region_rows):
    hints = []
    for pid in q["parent_ids"]:
        r = region_rows.get((q["source"], pid, q["book"]))
        if r and r.get("province"):
            hints.append(r["province"])
    names = list(q["parent_names"]) + ([q["name"]] if q["kind"] == "region" else [])
    for n in names:
        hints += REGION_NAME_HINTS.get(fold(n), [])
    hints += BOOK_PROVINCE_HINTS.get(q["book"], [])
    return list(dict.fromkeys(h for h in map(norm_province, hints) if h))


def make_row(q, win, considered, hints):
    row = {
        "query_id": q["query_id"], "source": q["source"], "source_id": q["source_id"], "book": q["book"],
        "name": q["name"], "aliases": "|".join(q["aliases"]), "kind": q["kind"],
        "year_min": q["year_min"], "year_max": q["year_max"], "year_source": q["year_source"],
        "n_mentions": q["n_mentions"], "parent_names": "|".join(q["parent_names"]),
        "status": "unmatched", "gazetteer": None, "time_basis": None, "feature_id": None, "feature_id_alt": None,
        "matched_name": None, "matched_via": None, "feature_type": None, "feature_class": None, "province": None,
        "latitude": None, "longitude": None, "geometry_type": None, "name_score": None, "score": None,
        "n_candidates": 0, "ambiguous": False, "runner_up_name": None, "runner_up_distance_km": None,
        "province_hints": "|".join(hints), "province_hint_met": None, "inherited_from": None, "_extra": {},
    }
    reasons = []
    if win is None:
        reasons.append("unmatched")
    else:
        gname, accepted = win
        best = accepted[0]
        row.update(status="matched", gazetteer=gname, time_basis=best.time_basis, feature_id=best.feature_id,
                   feature_id_alt=best.feature_id_alt, matched_name=best.name, matched_via=best.matched_via,
                   feature_type=best.feature_type, feature_class=best.feature_class, province=best.province,
                   latitude=best.lat, longitude=best.lon, geometry_type=best.geometry_type,
                   name_score=round(best.name_score, 3), score=round(best.score, 3), n_candidates=len(accepted),
                   _extra=best.extra)
        for c in accepted[1:]:
            same = c.feature_id == best.feature_id or (c.feature_id_alt and c.feature_id_alt == best.feature_id_alt)
            dist = haversine_km(best.lat, best.lon, c.lat, c.lon)
            if same or (dist is not None and dist <= AMBIGUITY_KM):
                continue
            if best.score - c.score <= AMBIGUITY_MARGIN:
                row.update(ambiguous=True, runner_up_name=c.name,
                           runner_up_distance_km=None if dist is None else round(dist, 1))
                reasons.append("ambiguous")
            break
        if best.matched_via == "alias":
            reasons.append("alias_match")
        elif best.matched_via == "variant":
            reasons.append("variant_match")
        elif best.name_score < 0.999:
            reasons.append("fuzzy_name")
        if hints:
            row["province_hint_met"] = best.province in hints
            if not row["province_hint_met"]:
                reasons.append("province_hint_unmet")
    if q["year_source"] == "unknown":
        reasons.append("year_unknown")
    row["_reasons"] = reasons
    return row


def candidate_log(q, considered, selected_row):
    out = []
    for gname, cands in considered:
        for rank, c in enumerate(cands[:MAX_CANDIDATES_LOGGED], start=1):
            out.append({
                "query_id": q["query_id"], "gazetteer": gname, "rank": rank, "feature_id": c.feature_id,
                "feature_id_alt": c.feature_id_alt, "name": c.name, "latitude": c.lat, "longitude": c.lon,
                "feature_type": c.feature_type, "feature_class": c.feature_class, "province": c.province,
                "name_score": round(c.name_score, 3), "score": round(c.score, 3), "matched_via": c.matched_via,
                "selected": selected_row["status"] == "matched" and selected_row["gazetteer"] == gname
                            and selected_row["feature_id"] == c.feature_id})
    return out


def run_matching(queries):
    rows, region_rows, log = {}, {}, []
    phases = [[q for q in queries if q["kind"] == "region"], [q for q in queries if q["kind"] != "region"]]
    for phase in phases:
        for q in tqdm(phase, desc="Matching"):
            hints = province_hints(q, region_rows)
            win, considered = resolve_one(q, hints)
            row = make_row(q, win, considered, hints)
            rows[q["query_id"]] = row
            log.extend(candidate_log(q, considered, row))
            if q["kind"] == "region":
                region_rows[(q["source"], q["source_id"], q["book"])] = row

    if INHERIT_FROM_PARENT:
        for q in queries:
            row = rows[q["query_id"]]
            if row["status"] != "unmatched":
                continue
            for pid in q["parent_ids"]:
                parent = region_rows.get((q["source"], pid, q["book"]))
                if parent and parent["latitude"] is not None:
                    row.update(status="inherited", gazetteer=parent["gazetteer"], time_basis=parent["time_basis"],
                               matched_name=parent["matched_name"], feature_class=parent["feature_class"],
                               province=parent["province"], latitude=parent["latitude"],
                               longitude=parent["longitude"], geometry_type="inherited", inherited_from=pid)
                    row["_reasons"] = [r for r in row["_reasons"] if r != "unmatched"] + ["inherited"]
                    break
    return [rows[q["query_id"]] for q in queries], log

### 4.3 Enrichment: census subdivision linkage

An enricher adds columns to matched rows after resolution. `chgis_csd` assigns a census subdivision from the layer nearest to the place's year:

- Rows matched by `chgis_name` take the subdivision they matched (`csd_method = name`).
- Other rows with a point and a feature class in `CSD_ELIGIBLE_CLASSES` are tested against the layer polygons (`point_in_polygon`). A point outside every polygon is assigned the nearest polygon within `CSD_SNAP_M` metres (`nearest`), otherwise `no_polygon`.
- Provinces, territories and countries, and water features, are `not_applicable`.

Adding an enricher: subclass `Enricher`, implement `apply(rows)` to add fields to the row dicts in place, and list it in `ENRICHERS`.

> **[DESIGN NOTE — delete before publishing]**
> **Containment is tested with the match's modern point against historical polygons.** The alternative, matching CSD names to the place name, only works when the place was itself a subdivision (a city, township or parish). A point-in-polygon test also covers villages, islands, institutions and landmarks that lie inside a subdivision, which is what a census join needs. Cost: a feature whose modern coordinate has moved, or a feature represented by a point inside a larger area, can land in the wrong subdivision. `csd_method` and `csd_gap_years` are recorded so that such rows can be filtered.
>
> **Snapping to the nearest polygon within `CSD_SNAP_M`.** CGNDB points for coastal features and islands often fall in water or on a boundary that the historical polygons do not cover. A strict test would reject them. The default of 3 km recovers these without assigning inland points to distant subdivisions; snapped rows are flagged `csd_snapped` for review. Set `CSD_SNAP_M = 0` to disable.
>
> **Large features are excluded.** Assigning a province or a river to one subdivision is meaningless. `CSD_ELIGIBLE_CLASSES` controls this and is editable.

In [ ]:
class Enricher:
    name = "base"

    def apply(self, rows):
        raise NotImplementedError


class CHGISContainment(Enricher):
    name = "chgis_csd"

    def __init__(self, layers):
        self.layers = layers

    def _blank(self):
        base = {"csd_status": "not_applicable", "csd_method": None, "csd_layer_year": None, "csd_gap_years": None,
                "csd_id": None, "csd_name": None, "csd_snap_m": None}
        base.update({c: None for c in self.layers.extra_columns()})
        return base

    def apply(self, rows):
        by_layer = defaultdict(list)
        for r in rows:
            r.update(self._blank())
            if r["status"] != "matched":
                continue
            if r["gazetteer"] == "chgis_name":
                ex = r["_extra"]
                r.update(csd_status="matched", csd_method="name", csd_layer_year=ex["layer_year"],
                         csd_gap_years=ex["gap"], csd_id=ex["csd_id"], csd_name=ex["csd_name"], **ex["fields"])
                continue
            if r["feature_class"] not in CSD_ELIGIBLE_CLASSES:
                continue
            if r["latitude"] is None:
                r["csd_status"] = "no_coordinates"
                continue
            if r["year_min"] is None:
                r["csd_status"] = "no_year"
                continue
            layer_year, gap = self.layers.pick(round((r["year_min"] + r["year_max"]) / 2))
            if layer_year is None:
                r.update(csd_status="out_of_period", csd_gap_years=gap)
                continue
            r["csd_gap_years"] = gap
            by_layer[layer_year].append(r)
        for layer_year, rs in by_layer.items():
            self._link_layer(layer_year, rs)

    def _link_layer(self, layer_year, rs):
        layer = self.layers.get(layer_year)
        cols = ["csd_id", "csd_name"] + self.layers.extra_columns()
        pts = gpd.GeoDataFrame({"k": range(len(rs))},
                               geometry=gpd.points_from_xy([r["longitude"] for r in rs], [r["latitude"] for r in rs]),
                               crs=4326).to_crs(layer.crs)
        joined = gpd.sjoin(pts, layer[cols + ["geometry"]], how="left", predicate="within")
        joined = joined[~joined.index.duplicated(keep="first")]
        method = {}
        for i in joined.index[joined["csd_id"].notna()]:
            method[i] = ("point_in_polygon", None, joined.loc[i])
        miss = pts.loc[[i for i in pts.index if i not in method]]
        if len(miss) and CSD_SNAP_M and layer.crs.is_projected:
            near = gpd.sjoin_nearest(miss, layer[cols + ["geometry"]], how="left",
                                     max_distance=CSD_SNAP_M, distance_col="snap_m")
            near = near[~near.index.duplicated(keep="first")]
            for i in near.index[near["csd_id"].notna()]:
                method[i] = ("nearest", float(near.loc[i, "snap_m"]), near.loc[i])
        for i, r in enumerate(rs):
            if i in method:
                how, snap, src = method[i]
                r.update(csd_status="matched", csd_method=how, csd_layer_year=layer_year,
                         csd_id=src["csd_id"], csd_name=src["csd_name"],
                         csd_snap_m=None if snap is None else round(snap, 1),
                         **{c: src[c] for c in self.layers.extra_columns()})
            else:
                r.update(csd_status="no_polygon", csd_layer_year=layer_year)


ENRICHER_OBJECTS = []
if "chgis_csd" in ENRICHERS:
    if CHGIS is not None:
        ENRICHER_OBJECTS.append(CHGISContainment(CHGIS))
    else:
        print("NOTE: enricher chgis_csd skipped, CHGIS not loaded")

## 5. Run

In [ ]:
ROWS, CANDIDATE_LOG = run_matching(QUERIES)
for enricher in ENRICHER_OBJECTS:
    enricher.apply(ROWS)

for r in ROWS:
    reasons = list(r.pop("_reasons"))
    if r.get("csd_method") == "nearest":
        reasons.append("csd_snapped")
    if r.get("csd_status") == "no_polygon":
        reasons.append("csd_no_polygon")
    r["review_reasons"] = "|".join(reasons)
    r["needs_review"] = bool(reasons)

print(pd.Series([r["status"] for r in ROWS]).value_counts().to_string())

## 6. Write outputs

In [ ]:
BASE_COLUMNS = ["query_id", "source", "source_id", "book", "name", "aliases", "kind", "year_min", "year_max",
                "year_source", "n_mentions", "parent_names", "status", "gazetteer", "time_basis", "feature_id",
                "feature_id_alt", "matched_name", "matched_via", "feature_type", "feature_class", "province",
                "latitude", "longitude", "geometry_type", "name_score", "score", "n_candidates", "ambiguous",
                "runner_up_name", "runner_up_distance_km", "province_hints", "province_hint_met", "inherited_from"]
CSD_COLUMNS = ["csd_status", "csd_method", "csd_layer_year", "csd_gap_years", "csd_id", "csd_name", "csd_snap_m"]
EXTRA_CSD = CHGIS.extra_columns() if CHGIS is not None and ENRICHER_OBJECTS else []
OUTPUT_COLUMNS = BASE_COLUMNS + (CSD_COLUMNS + EXTRA_CSD if ENRICHER_OBJECTS else []) + ["needs_review", "review_reasons"]

matches = pd.DataFrame([{c: r.get(c) for c in OUTPUT_COLUMNS} for r in ROWS], columns=OUTPUT_COLUMNS)
for c in ("year_min", "year_max", "n_mentions", "csd_layer_year", "csd_gap_years"):
    if c in matches:
        matches[c] = matches[c].astype("Int64")
candidates_df = pd.DataFrame(CANDIDATE_LOG)

matches.to_csv(MATCHES_CSV, index=False, encoding="utf-8")
candidates_df.to_csv(CANDIDATES_CSV, index=False, encoding="utf-8")
records = json.loads(matches.astype(object).where(matches.notna(), None).to_json(orient="records", force_ascii=False))
MATCHES_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=1), encoding="utf-8")

with_xy = [r for r in records if r["latitude"] is not None]
MATCHES_GEOJSON.write_text(json.dumps({"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [r["longitude"], r["latitude"]]},
     "properties": {k: v for k, v in r.items() if k not in ("latitude", "longitude")}} for r in with_xy]},
    ensure_ascii=False), encoding="utf-8")

# Polygons of the census subdivisions linked at least once
csd_frames = []
if ENRICHER_OBJECTS:
    linked = matches[matches["csd_status"] == "matched"].dropna(subset=["csd_layer_year"])
    for layer_year, g in linked.groupby("csd_layer_year"):
        layer = CHGIS.get(int(layer_year))
        sel = layer[layer["csd_id"].isin(set(g["csd_id"]))].copy()
        if CSD_GEOJSON_SIMPLIFY_M:
            sel["geometry"] = sel.geometry.simplify(CSD_GEOJSON_SIMPLIFY_M)
        sel["layer_year"] = int(layer_year)
        sel["queries"] = sel["csd_id"].map(g.groupby("csd_id")["name"].agg(lambda s: "|".join(sorted(set(s)))))
        csd_frames.append(sel.to_crs(4326))
if csd_frames:
    pd.concat(csd_frames).pipe(gpd.GeoDataFrame, crs=4326).to_file(CSD_GEOJSON, driver="GeoJSON")
else:
    CSD_GEOJSON.write_text(json.dumps({"type": "FeatureCollection", "features": []}), encoding="utf-8")

RUN_METADATA_JSON.write_text(json.dumps({
    "created": datetime.now().isoformat(timespec="seconds"),
    "active_inputs": ACTIVE_INPUTS, "books": BOOKS,
    "routes": ROUTES, "enrichers": ENRICHERS,
    "parameters": dict(MIN_NAME_SCORE=MIN_NAME_SCORE, FUZZY_ENABLED=FUZZY_ENABLED, USE_ALIASES=USE_ALIASES,
                       ALIAS_PENALTY=ALIAS_PENALTY, ARTICLE_PENALTY=ARTICLE_PENALTY, VARIANT_PENALTY=VARIANT_PENALTY,
                       RELEVANCE_WEIGHT=RELEVANCE_WEIGHT, PROVINCE_MATCH_BONUS=PROVINCE_MATCH_BONUS,
                       AMBIGUITY_MARGIN=AMBIGUITY_MARGIN, AMBIGUITY_KM=AMBIGUITY_KM, INHERIT_FROM_PARENT=INHERIT_FROM_PARENT,
                       CHGIS_MAX_GAP_YEARS=CHGIS_MAX_GAP_YEARS, CSD_SNAP_M=CSD_SNAP_M),
    "gazetteers": {g.name: dict(records=g.n_records, source=g.source, time_basis=g.time_basis) for g in GAZETTEERS.values()},
    "counts": {"queries": len(ROWS), "by_status": matches["status"].value_counts().to_dict(),
               "needs_review": int(matches["needs_review"].sum())},
}, indent=1, default=str), encoding="utf-8")

print(f"Wrote {len(matches)} rows  -> {MATCHES_CSV}")
print(f"Wrote {len(candidates_df)} rows  -> {CANDIDATES_CSV}")
print(f"Wrote {len(with_xy)} points -> {MATCHES_GEOJSON}")
print(f"Wrote {sum(len(f) for f in csd_frames)} polygons -> {CSD_GEOJSON}")

## 7. Diagnostics

Summary tables for judging the matches. The review table is the primary input for tuning `ROUTES`, the bonuses and the overrides.

In [ ]:
print("Status by book and kind")
print(pd.crosstab([matches["book"], matches["kind"].fillna("(unspecified)")], matches["status"], margins=True).to_string())
print("\nMatches by gazetteer")
print(matches.loc[matches["status"] == "matched", "gazetteer"].value_counts().to_string())

In [ ]:
print("Review reasons")
print(matches["review_reasons"].str.split("|").explode().replace("", np.nan).value_counts().to_string())

In [ ]:
if ENRICHER_OBJECTS:
    print("Census subdivision status")
    print(pd.crosstab(matches["csd_status"], matches["csd_method"].fillna("(none)"), margins=True).to_string())

In [ ]:
print("Unmatched places, most frequent first (candidates for overrides, other gazetteers, or story locations)")
matches[matches["status"] == "unmatched"].sort_values("n_mentions", ascending=False)[
    ["book", "name", "kind", "n_mentions", "parent_names", "year_min", "year_max"]].head(30)

In [ ]:
print("Ambiguous matches")
matches[matches["ambiguous"].fillna(False)][
    ["book", "name", "matched_name", "province", "latitude", "longitude", "runner_up_name", "runner_up_distance_km"]].head(30)

In [ ]:
print("Random sample of matches for spot-checking")
m = matches[matches["status"] == "matched"]
m.sample(min(20, len(m)), random_state=0)[
    ["book", "name", "gazetteer", "matched_name", "feature_type", "province", "name_score", "csd_name", "review_reasons"]]

## 8. Extension points

| Component | Contract | Registration |
|---|---|---|
| Input adapter | Function `f(books) -> list[dict]` returning query dicts with the fields in `QUERY_FIELDS` | `INPUT_ADAPTERS[name]`, then `ACTIVE_INPUTS` |
| Gazetteer | Subclass of `Gazetteer` with `name`, `time_basis` and `candidates(query, variants) -> list[Candidate]` | `build_gazetteers()`, then a route in `ROUTES` |
| Enricher | Subclass of `Enricher` with `apply(rows)` adding fields to row dicts in place | `ENRICHER_OBJECTS`, then `ENRICHERS` |
| Route | Dict with `gazetteer` and optional `kinds`, `books`, `year_from`, `year_to`, `min_name_score`, `require_year` | `ROUTES` |

Period-specific gazetteers are expressed as routes with year bounds. For example, a gazetteer of New France placed ahead of the CGNDB for early books:

```python
ROUTES = [
    dict(gazetteer="overrides"),
    dict(gazetteer="new_france", year_to=1763),
    dict(gazetteer="cgndb"),
    dict(gazetteer="chgis_name", require_year=True, year_from=1841, year_to=1931),
]
```